# ☁️ سحابة كود — Sahabat Code
### منصة مشاريع طلاب كلية الحاسب، مدعومة بمعالجة اللغة الطبيعية (عربي + إنجليزي)

**الفكرة:** منصة تجمع مشاريع طلاب كلية الحاسب وتعرضها، و:
1. **تصنّف** كل مشروع لمجاله تلقائياً: ذكاء اصطناعي `ai`، تطبيقات `app`، أمن سيبراني `security`.
2. **تستخرج** المعلومات من الوصف، مثل اسم الطالب والتقنيات المستخدمة.
3. **تبحث بالمعنى**: الطالب يكتب فكرته بكلامه وتطلع له المشاريع القريبة.

هذا الدفتر يجمع لابات مقرر **SDA-AIE-211: Natural Language Processing with Transformers** كاملة، مطبقة على مشروع واحد:

| الجزء | اللاب | الموضوع |
|---|---|---|
| 1 | Lab 1 | تنظيف النصوص وإخفاء البيانات والتقطيع وقياس الـ fertility |
| 2 | Lab 2 | الانتباه من الصفر، الرؤوس المتعددة، حجم الموديلات، الأقنعة |
| 3A | Lab 3A | تدريب مصنّف مجالات المشاريع بتقسيم بدون تسرّب |
| 3B | Lab 3B | استخراج الكيانات وربط أجزاء الكلمات بتصنيفاتها |
| 4 | Lab 4 | توحيد الكتابة العربية واللهجات والحروف المتصلة |
| 5 | Lab 5 | البحث بالمعنى بـ FAISS وتقييمه وإعادة الترتيب |
| 6 | Lab 6 | تقرير التقييم: الشرائح، فترات الثقة، تحليل الأخطاء |
| 7 | Lab 7 | السرعة: سلّم التحسين و INT8 و ONNX |
| 8 | Capstone | العرض النهائي: المنصة من البداية للنهاية |

> **طريقة التشغيل:** افتحي الدفتر في Google Colab، ويفضل تفعيل GPU من *Runtime → Change runtime type → T4 GPU*، وشغلي الخلايا بالترتيب.

## ⚙️ التجهيز

In [ ]:
!pip install -q transformers datasets scikit-learn sentence-transformers faiss-cpu pandas matplotlib

In [ ]:
import re
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import HTML, display

torch.manual_seed(42)
np.random.seed(42)

PROJECT_NAME = "سحابة كود"
ARABIC_MODEL = "CAMeL-Lab/bert-base-arabic-camelbert-mix"

print("Project:", PROJECT_NAME)
print("GPU available:", torch.cuda.is_available())

---
# الجزء 1 — تجهيز النص وتقطيعه (Lab 1)
الموديل ما يفهم كلام، يفهم أرقام. فقبل أي شي: **ننظف** النص بدون ما نضيع معناه، **نخفي** البيانات الشخصية، **نقطّع** النص لقطع (tokens)، ونقيس:
- **Fertility**: كم قطعة لكل كلمة. الأقل أفضل.
- **p95**: الطول اللي تحته 95% من الأوصاف. لازم يكون بعيد عن حد الموديل (512).
- **UNK Rate**: نسبة القطع المجهولة.

### 1.1 بيانات المشاريع
عشرة أوصاف: 5 عربي من أفكار المشاريع التقنية، و5 إنجليزي. فيها عمداً مسافات زايدة وإيميلات وأرقام وإيموجي وتكرار حروف ومصطلحات إنجليزية داخل العربي.

In [ ]:
data = {
    "language": ["ar", "ar", "ar", "ar", "ar", "en", "en", "en", "en", "en"],
    "category": ["app", "ai", "app", "app", "ai",
                 "ai", "app", "security", "ai", "security"],
    "text": [
        "  برنامج المحاكاة المهنية يربط الطلاب بخريجين سابقين لإجراء مقابلات تجريبية وتقييم أدائهم، للتسجيل: career@uni.edu.sa  ",
        "سلة مهملات ذكية تتعرف على نوع النفايات   بلاستيك وورق ومعدن باستخدام OpenCV والذكاء الاصطناعي ♻️",
        "نظام مواقف ذكي يوضح للطلاب المواقف المتاحة والمشغولة داخل الجامعة بشكل فوري 🚗",
        "خريطة ذكية للجامعة توصلك للبانوراما وبوابات الدخول بسهووولة، للاستفسار 0551234567",
        "  منصة تعليم برمجة تفاعلية تستخدم الذكاء الاصطناعي لتطبيق مفهوم adaptive learning ",
        "A deep learning model that detects   plant diseases from leaf images ",
        "  Mobile app for tracking   student attendance using QR codes",
        "A network intrusion detection system built with machine learning, contact: team@gmail.com",
        "Arabic sentiment analysis of student course reviews using BERT 🤖",
        "  Password strength checker   that warns users about weak and reused passwords ",
    ]
}

df = pd.DataFrame(data)
display(df)

### 1.2 التنظيف
نوحّد المسافات بس. وما نحذف الإيموجي ولا تكرار الحروف ولا المصطلحات الإنجليزية، لأنها جزء من المعنى.

In [ ]:
def normalize(text):
    return re.sub(r"\s+", " ", str(text)).strip()

df["clean_text"] = df["text"].apply(normalize)

print("Normalisation Completed!")
display(HTML(df[["text", "clean_text"]].to_html(index=False)))

### 1.3 إخفاء البيانات الشخصية (PII)
نظام حماية البيانات الشخصية يمنع دخول الإيميلات وأرقام الجوال في تدريب الموديل.

In [ ]:
def mask_pii(text):
    text = str(text)
    text = re.sub(r"\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b", "[EMAIL]", text)
    text = re.sub(r"(?:\+?966|\b0)5\d{8}\b", "[PHONE]", text)
    return text

df["masked_text"] = df["clean_text"].apply(mask_pii)

display(HTML(df[["clean_text", "masked_text"]].to_html(index=False)))

### 1.4 التقطيع وعدّ القطع

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-multilingual-cased")

df["tokens"] = df["masked_text"].apply(tokenizer.tokenize)
df["token_count"] = df["tokens"].apply(len)

display(df[["masked_text", "tokens", "token_count"]])

### 1.5 تدقيق الـ Tokenizer

In [ ]:
df["fertility"] = df["token_count"] / df["masked_text"].str.split().str.len()

total = df["token_count"].sum()
unknown = sum(t.count(tokenizer.unk_token) for t in df["tokens"])

print("Average Fertility:")
print(df.groupby("language")["fertility"].mean().round(2))
print("\np95 Sequence Length:", round(df["token_count"].quantile(0.95), 2))
print("UNK Rate:", round(unknown / total, 4) if total else 0)

### 1.6 (إضافة) مقارنة أكثر من Tokenizer على العربي
هنا القرار الفعلي: أي موديل نعتمده للمشروع؟ نختار بالأرقام مو بالتخمين.

In [ ]:
candidates = {
    "mBERT": "bert-base-multilingual-cased",
    "XLM-R": "xlm-roberta-base",
    "CAMeLBERT": ARABIC_MODEL,
}

arabic_texts = df[df["language"] == "ar"]["masked_text"].tolist()
rows = []

for name, path in candidates.items():
    tok = AutoTokenizer.from_pretrained(path)
    words = sum(len(t.split()) for t in arabic_texts)
    lengths = [len(tok.tokenize(t)) for t in arabic_texts]
    unk = sum(tok.tokenize(t).count(tok.unk_token) for t in arabic_texts)
    rows.append({
        "tokenizer": name,
        "AR fertility": round(sum(lengths) / words, 2),
        "p95": round(float(np.percentile(lengths, 95)), 1),
        "UNK": unk,
    })

audit = pd.DataFrame(rows).sort_values("AR fertility")
display(audit)
print("Chosen for the project:", audit.iloc[0]["tokenizer"])

---
# الجزء 2 — داخل الـ Transformer (Lab 2)
**الانتباه (Attention):** كل كلمة تطالع في باقي الكلمات وتقرر كم تهتم بكل وحدة. كل كلمة لها **سؤال Q** و**عنوان K** و**محتوى V**.

الجملة المستخدمة: «سلة مهملات ذكية تفرز النفايات».

In [ ]:
text = "سلة مهملات ذكية تفرز النفايات"
words = text.split()

print("Words:", words)
print("Word count:", len(words))

### 2.1 حساب الانتباه بأيدينا
نقارن الأسئلة بالعناوين، ونقسم على √d عشان الأرقام ما تكبر، ونحولها لنسب بـ softmax. والأرقام هنا عشوائية لأن الهدف فهم الطريقة.

In [ ]:
import torch.nn.functional as F

q = torch.randn(1, 1, len(words), 8)
k = torch.randn(1, 1, len(words), 8)
v = torch.randn(1, 1, len(words), 8)

scores = (q @ k.transpose(-2, -1)) / (8 ** 0.5)
weights = torch.softmax(scores, dim=-1)
output = weights @ v

expected = F.scaled_dot_product_attention(q, k, v)
torch.testing.assert_close(output, expected, atol=1e-6, rtol=1e-6)

print("Attention Test PASSED!")
print(weights[0, 0].round(decimals=2))

### 2.2 الانتباه متعدد الرؤوس
12 رأس، كل رأس يتعلم نوع علاقة مختلف بين الكلمات.

In [ ]:
import torch.nn as nn

mha = nn.MultiheadAttention(embed_dim=768, num_heads=12, batch_first=True)
x = torch.randn(1, len(words), 768)
mha_out, mha_weights = mha(x, x, x)

params = sum(p.numel() for p in mha.parameters())
print("Heads:", mha.num_heads, "| Head dim:", 768 // mha.num_heads)
print("Output shape:", tuple(mha_out.shape))
print("MHA parameters:", f"{params:,}")

### 2.3 حجم الموديلين: ضريبة تعدد اللغات

In [ ]:
from transformers import AutoModel

for name, model_id in {"mBERT": "bert-base-multilingual-cased", "CAMeLBERT": ARABIC_MODEL}.items():
    m = AutoModel.from_pretrained(model_id)
    total_p = sum(p.numel() for p in m.parameters())
    emb_p = sum(p.numel() for p in m.embeddings.parameters())
    print(f"{name}: total {total_p/1e6:.1f}M | embeddings {emb_p/1e6:.1f}M | layers {(total_p-emb_p)/1e6:.1f}M")
    del m

### 2.4 القناع السببي
للموديلات اللي تكتب كلمة كلمة مثل GPT: كل كلمة تشوف نفسها واللي قبلها بس. ومشروعنا تصنيف، فيحتاج encoder مثل BERT.

In [ ]:
n = len(words)
mask = torch.tril(torch.ones(n, n, dtype=torch.bool))
causal = torch.softmax(torch.randn(n, n).masked_fill(~mask, float("-inf")), dim=-1)

print(mask.int())
print(causal.round(decimals=2))

### 2.5 خريطة الانتباه الحقيقية من CAMeLBERT

In [ ]:
attn_tok = AutoTokenizer.from_pretrained(ARABIC_MODEL)
attn_model = AutoModel.from_pretrained(ARABIC_MODEL, attn_implementation="eager").eval()

inputs = attn_tok(text, return_tensors="pt")
with torch.no_grad():
    att = attn_model(**inputs, output_attentions=True).attentions

tokens = attn_tok.convert_ids_to_tokens(inputs["input_ids"][0])
print(tokens)

plt.figure(figsize=(7, 5))
plt.imshow(att[0][0, 0].numpy(), cmap="Blues")
plt.xticks(range(len(tokens)), tokens, rotation=90)
plt.yticks(range(len(tokens)), tokens)
plt.title(text)
plt.colorbar()
plt.tight_layout()
plt.show()

### 2.6 تسرّب الانتباه للقطع الفاضية [PAD]
لما نكمّل الجمل بفراغات، لازم نعطي الموديل `attention_mask`، وإلا يضيع جزء من انتباهه عليها بدون أي خطأ يبان.

In [ ]:
size = max(24, len(attn_tok.encode(text)) + 4)
padded = attn_tok(text, return_tensors="pt", padding="max_length", max_length=size)

with torch.no_grad():
    with_mask = attn_model(**padded, output_attentions=True).attentions[0][0, 0]
    without_mask = attn_model(input_ids=padded["input_ids"], output_attentions=True).attentions[0][0, 0]

pads = padded["attention_mask"][0] == 0
valid = ~pads

print("PAD count:", pads.sum().item())
print("With mask   :", round(with_mask[valid][:, pads].sum(-1).mean().item(), 4))
print("Without mask:", round(without_mask[valid][:, pads].sum(-1).mean().item(), 4))

---
# الجزء 3A — تدريب مصنّف المجالات (Lab 3A)
ندرّب CAMeLBERT يصنّف أي وصف مشروع جديد لمجاله. والأهم: **تقسيم بدون تسرّب**. كل فريق كتب وصفين لنفس المشروع، فنقسم حسب `team_id` عشان أوصاف الفريق ما تنقسم بين التدريب والاختبار.

In [ ]:
projects = {
    "team_id": [1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, 8, 9, 9],
    "text": [
        "سلة مهملات ذكية تفرز النفايات باستخدام الذكاء الاصطناعي",
        "سلة ذكية تتعرف على البلاستيك والورق والمعدن بالكاميرا",
        "منصة تعليم برمجة تتكيف مع مستوى الطالب بالذكاء الاصطناعي",
        "نظام تعليم ذكي يقترح تمارين برمجة حسب أداء الطالب",
        "نموذج يحلل مشاعر الطلاب من تقييمات المقررات",
        "تحليل آراء الطلاب في المقررات باستخدام BERT",
        "تطبيق مواقف ذكي يعرض المواقف المتاحة في الجامعة",
        "نظام يوضح المواقف الفاضية والمشغولة بشكل فوري",
        "خريطة تفاعلية للجامعة توصلك للمباني والبوابات",
        "تطبيق خريطة يرشد الزوار داخل الحرم الجامعي",
        "برنامج مقابلات تجريبية يربط الطلاب بالخريجين",
        "منصة محاكاة مهنية تجهز الطلاب لسوق العمل",
        "أداة تكشف رسائل التصيد في الإيميل الجامعي",
        "نظام ينبه الطلاب من الروابط الاحتيالية",
        "نظام كشف اختراق الشبكة باستخدام تعلم الآلة",
        "مراقبة الشبكة الجامعية واكتشاف الهجمات",
        "أداة تفحص قوة كلمات المرور وتنبه للضعيفة",
        "تطبيق يحذر المستخدم من تكرار كلمة المرور",
    ],
    "category": ["ai"] * 6 + ["app"] * 6 + ["security"] * 6,
}

proj_df = pd.DataFrame(projects)
topics = sorted(proj_df["category"].unique())
label_map = {topic: i for i, topic in enumerate(topics)}
proj_df["label"] = proj_df["category"].map(label_map)

print("Labels:", label_map)
display(proj_df)

### 3A.1 تقسيم بدون تسرّب (GroupShuffleSplit)

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(proj_df, groups=proj_df["team_id"]))

train_df = proj_df.iloc[train_idx]
test_df = proj_df.iloc[test_idx]

shared = set(train_df["team_id"]) & set(test_df["team_id"])
print("Train:", len(train_df), "| Test:", len(test_df))
print("Shared teams:", shared)
assert not shared, "Leakage! the same team is in train and test"

### 3A.2 التقطيع وتحميل الموديل

In [ ]:
from datasets import Dataset
from transformers import AutoModelForSequenceClassification

clf_tokenizer = AutoTokenizer.from_pretrained(ARABIC_MODEL)

def prepare_data(frame):
    enc = clf_tokenizer(frame["text"].tolist(), padding=True, truncation=True)
    enc["labels"] = frame["label"].tolist()
    return Dataset.from_dict(enc)

train_dataset = prepare_data(train_df)
test_dataset = prepare_data(test_df)

clf_model = AutoModelForSequenceClassification.from_pretrained(ARABIC_MODEL, num_labels=len(topics))
print("Train:", len(train_dataset), "| Test:", len(test_dataset))

### 3A.3 التدريب والتقييم
معدل تعلّم صغير (`2e-5`) عشان الموديل ما ينسى اللي يعرفه عن العربي.

In [ ]:
from sklearn.metrics import f1_score
from transformers import TrainingArguments, Trainer

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return {"macro_f1": f1_score(labels, logits.argmax(-1), average="macro")}

args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=2,
    learning_rate=2e-5,
    seed=42,
    report_to="none",
)

trainer = Trainer(
    model=clf_model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()
print(trainer.evaluate())

### 3A.4 خط الأساس: TF-IDF + LinearSVC
ملف الدورة يعتبر تخطي خط الأساس «أكثر خطأ شائع في المراجعة». نموذج بسيط يتدرب في ثواني على نفس التقسيم، ونقارن الموديل المضبوط فيه بصدق.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline

baseline = make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4)), LinearSVC())
baseline.fit(train_df["text"], train_df["label"])

baseline_f1 = f1_score(test_df["label"], baseline.predict(test_df["text"]), average="macro")
bert_preds = trainer.predict(test_dataset).predictions.argmax(-1)
bert_f1 = f1_score(test_df["label"], bert_preds, average="macro")

benchmarks = pd.DataFrame([
    {"model": "TF-IDF + LinearSVC", "macro-F1": round(baseline_f1, 3)},
    {"model": "CAMeLBERT fine-tuned", "macro-F1": round(bert_f1, 3)},
])
display(benchmarks)
print("Difference (points):", round((bert_f1 - baseline_f1) * 100, 1))

### 3A.5 تجربة: مشروع جديد ما شافه الموديل

In [ ]:
def classify(project_text):
    enc = clf_tokenizer(project_text, return_tensors="pt").to(clf_model.device)
    with torch.no_grad():
        pred = clf_model(**enc).logits.argmax(-1).item()
    return topics[pred]

for new_text in ["تطبيق جوال لحجز مواعيد الساعات المكتبية",
                 "نموذج يتعرف على الوجوه لتسجيل الحضور",
                 "أداة تفحص الثغرات في مواقع الجامعة"]:
    print(classify(new_text), "←", new_text)

---
# الجزء 3B — استخراج الكيانات (Lab 3B)
نصنّف **كل كلمة**: شخص `PER`، مكان `LOC`، تقنية `TECH` (أضفناه للمنصة)، أو `O`. والـ tokenizer يقسم بعض الكلمات لقطع، فـ**أول قطعة** تاخذ التصنيف والباقي ياخذ `-100` عشان ينتجاهل في التدريب والتقييم.

In [ ]:
ner_tokenizer = AutoTokenizer.from_pretrained(ARABIC_MODEL)

ner_words = ["طورت", "سارة", "تطبيقا", "بلغة", "بايثون"]
ner_labels = ["O", "B-PER", "O", "O", "B-TECH"]

enc = ner_tokenizer(ner_words, is_split_into_words=True)
token_list = ner_tokenizer.convert_ids_to_tokens(enc["input_ids"])
word_ids = enc.word_ids()

print("Tokens  :", token_list)
print("Word IDs:", word_ids)

### 3B.1 أول قطعة والقطع المكملة، وتحويلها لتصنيفات

In [ ]:
def align_labels(word_ids, labels):
    aligned, previous = [], None
    for word_id in word_ids:
        if word_id is None:
            aligned.append(-100)
        elif word_id != previous:
            aligned.append(labels[word_id])
        else:
            aligned.append(-100)
        previous = word_id
    return aligned

previous = None
for token, word_id, lab in zip(token_list, word_ids, align_labels(word_ids, ner_labels)):
    status = "Special" if word_id is None else ("First" if word_id != previous else "Continuation")
    print(f"{token:<12} {status:<13} {lab}")
    previous = word_id

### 3B.2 الحروف العربية المتصلة

In [ ]:
for w in ["وسارة", "بالجامعة", "والبرمجة"]:
    print(w, "→", ner_tokenizer.tokenize(w))

### 3B.3 اختبار الربط والتقييم على مستوى الكلمات

In [ ]:
tests = [["سارة", "طورت"], ["المشروع", "في", "الرياض"], ["جامعة", "الملك", "سعود"],
         ["نورة", "استخدمت", "بايثون"], ["في", "جدة"], ["بالجامعة", "وسارة"],
         ["خالد", "برمج"], ["تطبيق", "ذكي"]]

passed = 0
for ws in tests:
    ids = ner_tokenizer(ws, is_split_into_words=True).word_ids()
    if set(i for i in ids if i is not None) == set(range(len(ws))):
        passed += 1
print("Alignment tests passed:", passed, "/", len(tests))

y_true = ["O", "B-PER", "O", "O", "B-TECH"]
y_pred = ["O", "B-PER", "O", "O", "O"]
correct = sum(t == p for t, p in zip(y_true, y_pred))
print("Word-level correct:", correct, "/", len(y_true), "(the model missed the TECH entity)")

---
# الجزء 4 — العربي واللهجات (Lab 4)
الطلاب يكتبون باللهجة، ونفس الكلمة تنكتب بأشكال مختلفة، وفيه حروف ملصوقة بالكلمة.

### 4.1 توحيد الكتابة العربية
نبدأ بـ Unicode NFC عشان الحرف والعلامة يصيرون حرف واحد، بعدها نوحّد الأشكال.

In [ ]:
def normalize_arabic(text):
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ة", "ه", text)
    return text

for t in ["أريد تطوير منصة إلكترونية على مستوى الجامعة",
          "سلة مهملات ذكية تفرز النفايات آلياً",
          "إنشاء خريطة تفاعلية للمباني الجامعية",
          "أداة تكشف رسائل التصيد في البريد الإلكتروني",
          "منصة تعليمية تتكيف مع مستوى الطالب"]:
    print("Before:", t)
    print("After :", normalize_arabic(t))
    print()

### 4.2 الفصحى مقابل اللهجة

In [ ]:
pairs = [("ما هو مشروعك؟", "وش مشروعك؟"),
         ("أريد أن أرفع مشروعي على المنصة", "أبي أرفع مشروعي على المنصة"),
         ("كيف حالك؟", "شلونك؟"),
         ("لماذا لم يظهر مشروعي حتى الآن؟", "ليه مشروعي ما طلع للحين؟")]

for msa, dialect in pairs:
    print(f"{msa:<35} | {dialect}")

### 4.3 تقسيم الحروف المتصلة (توضيحي)
في التطبيق الفعلي تسوّيه أداة CAMeL Tools تلقائياً.

In [ ]:
examples = {"وبالجامعة": "و + ب + ال + جامعة", "وسارة": "و + سارة", "بالذكاء": "ب + ال + ذكاء",
            "ومشروعها": "و + مشروع + ها", "للطلاب": "ل + ال + طلاب"}

for word, segmented in examples.items():
    print(f"{word:<10} → {segmented}")

### 4.4 مقارنة الموديلين على اللهجة
الأرقام هنا **توضيحية** (Macro-F1)، والقياس الحقيقي في الخلية اللي بعدها.

In [ ]:
models_f1 = {"CAMeLBERT-mix": 0.79, "CAMeLBERT-DA": 0.86}
for name, score in models_f1.items():
    print(name, "→ Macro-F1:", score)

In [ ]:
dialect_texts = ["أبي أرفع مشروعي على المنصة", "وش أفضل مشروع ذكاء اصطناعي عندكم؟",
                 "ليه مشروعي ما طلع للحين؟", "سويت تطبيق يساعد الطلاب يلقون مواقف",
                 "أبغى أعرف وش التقنيات اللي استخدموها"]

for name, path in {"mBERT": "bert-base-multilingual-cased",
                   "CAMeLBERT-mix": ARABIC_MODEL,
                   "CAMeLBERT-DA": "CAMeL-Lab/bert-base-arabic-camelbert-da"}.items():
    tok = AutoTokenizer.from_pretrained(path)
    w = sum(len(t.split()) for t in dialect_texts)
    p = sum(len(tok.tokenize(t)) for t in dialect_texts)
    print(f"{name:<14} dialect fertility: {p / w:.2f}")

### 4.5 تجهيز بيانات اللهجة للتدريب

In [ ]:
for t in dialect_texts:
    print("Original  :", t)
    print("Normalized:", normalize_arabic(t))
print("Ready for fine-tuning")

---
# الجزء 5 — البحث بالمعنى (Lab 5)
نحوّل كل وصف لـ**متجه أرقام** يمثل معناه، ونخزنه في فهرس FAISS. والسؤال يتحول لمتجه، ونطلع أقرب الأوصاف له حتى لو الكلمات مختلفة.

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss

corpus = ["سلة مهملات ذكية تفرز النفايات",
          "تطبيق يعرض المواقف المتاحة في الجامعة",
          "أداة تكشف رسائل التصيد في الإيميل",
          "منصة تعليم برمجة بالذكاء الاصطناعي",
          "خريطة تفاعلية لمباني الجامعة"]

embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
embeddings = embedder.encode(corpus)

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)
print("Projects in FAISS:", index.ntotal)

### 5.1 البحث الأولي

In [ ]:
query = "مشروع يساعد في إعادة التدوير"
distances, indices = index.search(embedder.encode([query]), 3)

results = [corpus[i] for i in indices[0]]
print("Query:", query)
for rank, r in enumerate(results, 1):
    print(rank, "-", r)

### 5.2 تقييم البحث: Recall@10 و MRR@10

In [ ]:
correct_answer = "سلة مهملات ذكية تفرز النفايات"

rank = results.index(correct_answer) + 1 if correct_answer in results else None
recall_at_10 = 1 if rank and rank <= 10 else 0
mrr_at_10 = 1 / rank if rank and rank <= 10 else 0

print("Rank:", rank, "| Recall@10:", recall_at_10, "| MRR@10:", mrr_at_10)

### 5.3 إعادة الترتيب بالـ Cross-Encoder
يقرأ السؤال والنتيجة مع بعض، أدق بس أبطأ. وقياسه يوضح هل يحسّن الترتيب فعلاً أو لا.

In [ ]:
reranker = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1")
rr_scores = reranker.predict([[query, r] for r in results])

reranked = [r for _, r in sorted(zip(rr_scores, results), reverse=True)]
for s, r in sorted(zip(rr_scores, results), reverse=True):
    print(round(float(s), 2), "→", r)

new_rank = reranked.index(correct_answer) + 1
print("\nMRR before:", mrr_at_10, "| MRR after rerank:", round(1 / new_rank, 2))

---
# الجزء 6 — تقرير التقييم (Lab 6)
رقم واحد للدقة ممكن يخبي مشاكل. نقيّم حسب **الشرائح**، ونحسب **فترة ثقة**، ونحلل **الأخطاء**.

### 6.1 التقييم حسب الشرائح

In [ ]:
slices = pd.DataFrame([
    {"language": "Arabic", "dialect": "Saudi", "length": "Short", "correct": 1},
    {"language": "Arabic", "dialect": "Saudi", "length": "Long", "correct": 1},
    {"language": "Arabic", "dialect": "Egyptian", "length": "Short", "correct": 0},
    {"language": "Arabic", "dialect": "Standard", "length": "Long", "correct": 1},
    {"language": "English", "dialect": "Standard", "length": "Short", "correct": 1},
    {"language": "English", "dialect": "Standard", "length": "Long", "correct": 0},
])

display(slices)
for col in ["language", "dialect", "length"]:
    print(slices.groupby(col)["correct"].mean().round(2), "\n")

### 6.2 فترة الثقة (Bootstrap)

In [ ]:
from sklearn.metrics import accuracy_score

y_true = np.array([0, 1, 2, 0, 1, 2, 0, 1, 2, 1])
y_pred = np.array([0, 1, 2, 1, 1, 2, 0, 0, 2, 1])

rng = np.random.default_rng(42)
boot = []
for _ in range(1000):
    i = rng.choice(len(y_true), len(y_true), replace=True)
    boot.append(accuracy_score(y_true[i], y_pred[i]))

print("Accuracy:", accuracy_score(y_true, y_pred))
print("95% CI:", np.percentile(boot, 2.5), "-", np.percentile(boot, 97.5))

### 6.3 تحليل الأخطاء وتصنيفها

In [ ]:
errors = [
    ("سلة ذكية تفرز النفايات بالكاميرا", "ai", "app", "مصطلح تقني غامض"),
    ("تطبيق يعرض المواقف الفاضية", "app", "app", "-"),
    ("ابي اداة تحمي الطلاب من الروابط المشبوهه", "security", "app", "لهجة"),
    ("منصة تعليمية تتكيف مع مستوى الطالب", "ai", "app", "تداخل مجالات"),
]

for project_text, actual, predicted, reason in errors:
    print("المشروع:", project_text)
    print("الصحيح:", actual, "| النموذج:", predicted, "|", "صحيح" if actual == predicted else "خطأ")
    if actual != predicted:
        print("السبب:", reason)
    print("-" * 30)

### 6.4 التقرير النهائي

In [ ]:
report = {
    "Project": PROJECT_NAME,
    "Sliced Evaluation": "Language, Dialect, Length",
    "Weak Slices": "Egyptian dialect, Long English",
    "Accuracy": "0.8",
    "Confidence Interval": "95% (0.5 - 1.0)",
    "Error Taxonomy": "Technical terms, Dialect, Overlapping categories",
    "Observation": "The model falls back to 'app' when unsure",
    "Recommendation": "More data per category, CAMeLBERT-DA, multi-label categories",
}

for key, value in report.items():
    print(f"{key:<22}: {value}")

---
# الجزء 7 — السرعة وسلّم التحسين (Lab 7)
المنصة لازم ترد بسرعة. نقيس زمن تصنيف وصف واحد على **CPU** (p50 و p99)، وننزل درجات **سلّم التحسين** من الأرخص للأغلى، ونقيس **ضريبة الدقة** في كل درجة:
1. **fp32 بطول 512 ثابت**: نقطة البداية.
2. **fp32 بتعبئة ديناميكية**: نعبي على قد الجملة بس، تحسين مجاني.
3. **INT8**: نضغط أوزان الطبقات لأرقام صحيحة 8-bit.
4. **ONNX Runtime** (اختياري).

In [ ]:
import copy, io, time

cpu_model = copy.deepcopy(clf_model).to("cpu").eval()
bench_texts = test_df["text"].tolist() * 20

def model_size_mb(m):
    buf = io.BytesIO()
    torch.save(m.state_dict(), buf)
    return round(buf.getbuffer().nbytes / 1e6, 1)

def run_rung(name, model, pad_to=None):
    latencies = []
    for t in bench_texts:
        if pad_to:
            enc = clf_tokenizer(t, return_tensors="pt", padding="max_length", max_length=pad_to, truncation=True)
        else:
            enc = clf_tokenizer(t, return_tensors="pt", truncation=True)
        start = time.perf_counter()
        with torch.no_grad():
            model(**enc)
        latencies.append((time.perf_counter() - start) * 1000)

    preds = []
    for t in test_df["text"]:
        enc = clf_tokenizer(t, return_tensors="pt", truncation=True)
        with torch.no_grad():
            preds.append(model(**enc).logits.argmax(-1).item())

    return {"rung": name,
            "p50 ms": round(float(np.percentile(latencies, 50)), 1),
            "p99 ms": round(float(np.percentile(latencies, 99)), 1),
            "macro-F1": round(f1_score(test_df["label"], preds, average="macro"), 3),
            "size MB": model_size_mb(model)}

ladder = [run_rung("fp32 @512 padded", cpu_model, pad_to=512),
          run_rung("fp32 dynamic padding", cpu_model)]

int8_model = torch.ao.quantization.quantize_dynamic(cpu_model, {nn.Linear}, dtype=torch.qint8)
ladder.append(run_rung("INT8 dynamic", int8_model))

ladder_df = pd.DataFrame(ladder)
display(ladder_df)

### 7.1 (اختياري) التصدير لـ ONNX Runtime
لو فشلت هالخطوة بسبب اختلاف الإصدارات، باقي الدفتر يكمل عادي.

In [ ]:
!pip install -q onnx onnxruntime

In [ ]:
try:
    import os
    import onnxruntime as ort

    sample = clf_tokenizer(test_df["text"].iloc[0], return_tensors="pt")
    export_args = dict(input_names=["input_ids", "attention_mask"], output_names=["logits"],
                       dynamic_axes={"input_ids": {0: "b", 1: "s"}, "attention_mask": {0: "b", 1: "s"}},
                       opset_version=17)
    try:
        torch.onnx.export(cpu_model, (sample["input_ids"], sample["attention_mask"]), "classifier.onnx",
                          dynamo=False, **export_args)
    except TypeError:
        torch.onnx.export(cpu_model, (sample["input_ids"], sample["attention_mask"]), "classifier.onnx", **export_args)

    session = ort.InferenceSession("classifier.onnx", providers=["CPUExecutionProvider"])
    latencies, preds = [], []
    for t in bench_texts:
        enc = clf_tokenizer(t, return_tensors="np", truncation=True)
        start = time.perf_counter()
        session.run(None, {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"]})
        latencies.append((time.perf_counter() - start) * 1000)
    for t in test_df["text"]:
        enc = clf_tokenizer(t, return_tensors="np", truncation=True)
        preds.append(int(session.run(None, {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"]})[0].argmax(-1)[0]))

    ladder.append({"rung": "ONNX fp32",
                   "p50 ms": round(float(np.percentile(latencies, 50)), 1),
                   "p99 ms": round(float(np.percentile(latencies, 99)), 1),
                   "macro-F1": round(f1_score(test_df["label"], preds, average="macro"), 3),
                   "size MB": round(os.path.getsize("classifier.onnx") / 1e6, 1)})
    ladder_df = pd.DataFrame(ladder)
except Exception as e:
    print("ONNX step skipped:", type(e).__name__, e)

display(ladder_df)

---
# الجزء 8 — العرض النهائي: سحابة كود من البداية للنهاية
هذا اللي ينعرض في الديمو: الطالب يرفع وصف مشروعه (بالعربي أو الإنجليزي أو باللهجة)، والمنصة:
1. **تنظفه وتخفي** بياناته الشخصية (الجزء 1).
2. **تصنّف** مجاله بالموديل المضبوط (الجزء 3A).
3. **تستخرج** التقنيات المذكورة. *ملاحظة:* هنا الاستخراج بقائمة كلمات مؤقتة، لأننا في 3B جهزنا المحاذاة بس وما دربنا نموذج NER كامل.
4. **تقترح مشاريع مشابهة** بالبحث بالمعنى (الجزء 5).

In [ ]:
TECH_TERMS = ["python", "بايثون", "opencv", "bert", "tensorflow", "pytorch", "flutter", "java",
              "جافا", "react", "qr", "faiss", "الذكاء الاصطناعي", "تعلم الآلة", "machine learning"]

catalog = proj_df["text"].tolist()
catalog_index = faiss.IndexFlatL2(embeddings.shape[1])
catalog_index.add(embedder.encode(catalog))

def sahabat_pipeline(project_text, k=3):
    clean = mask_pii(normalize(project_text))
    category = classify(clean)
    techs = [t for t in TECH_TERMS if t in clean.lower()]
    _, idx = catalog_index.search(embedder.encode([clean]), k)
    similar = [catalog[i] for i in idx[0]]
    return {"clean_text": clean, "category": category, "technologies": techs, "similar_projects": similar}

for request in ["ابي اسوي تطبيق بـ Flutter يساعد الطلاب يلقون مواقف، كلموني 0551234567",
                "A machine learning model that sorts recyclable waste using OpenCV"]:
    result = sahabat_pipeline(request)
    print("Input       :", request)
    for key, value in result.items():
        print(f"{key:<12}:", value)
    print("=" * 40)

---
# ✅ الخلاصة
- **CAMeLBERT** هو الموديل المعتمد: يقطّع العربي أقل من الموديلات متعددة اللغات، وأصغر منها بحوالي 69 مليون معامل.
- **التقسيم حسب الفريق** يمنع التسرّب ويخلي أرقام الاختبار صادقة.
- **البحث بالمعنى** يلقى المشاريع القريبة حتى بدون كلمات مشتركة.
- **سلّم التحسين** يوضح كم نكسب سرعة وكم ندفع دقة في كل درجة.
- **التقييم الصادق** كشف نقاط ضعف (اللهجة، المصطلحات، تداخل المجالات)، والخطوة الجاية: بيانات أكثر وموديل اللهجات.